# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. `[1. 2. 3.] [0 1 1]`: `/` делит каждый элемент и возвращает дробные числа, `//` округляет частное вниз.
2. `[4. 4. 4.]`: `axis=1` суммирует четыре единицы внутри каждой из трёх строк.
3. `[[0, 1, 2], [1, 2, 3], [2, 3, 4]]`: формы `(3,)` и `(3, 1)` расширяются до `(3, 3)`; складываются значения строки и столбца.
4. `6.0`: корни равны `[1., 2., 3.]`, их сумма равна 6.
5. `7.5`: среднее квадратов — `(1 + 4 + 9 + 16) / 4`.

In [ ]:
# Проверка ответов на чтение кода
import numpy as np
a = np.array([2, 4, 6]); print(a / 2, a // 4)
m = np.ones((3, 4)); print(m.sum(axis=1))
print(np.arange(3) + np.arange(3).reshape(3, 1))
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
print((np.arange(1, 5) ** 2).mean())


[1. 2. 3.] [0 1 1]
[4. 4. 4.]
[[0 1 2]
 [1 2 3]
 [2 3 4]]
6.0
7.5


In [ ]:
from pathlib import Path
data_path = Path("orders.csv")
if not data_path.exists():
    candidates = list(Path(".").glob("orders*.csv"))
    if len(candidates) == 1:
        data_path = candidates[0]
    else:
        try:
            from google.colab import files
        except ImportError as exc:
            raise FileNotFoundError("Распакуйте архив и положите orders.csv рядом с ноутбуком") from exc
        uploaded = files.upload()
        candidates = [Path(name) for name in uploaded if name.lower().endswith(".csv")]
        if len(candidates) != 1:
            raise ValueError("Загрузите один CSV-файл с заказами")
        data_path = candidates[0]
print("Файл данных:", data_path.name)

Файл данных: orders.csv


In [ ]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open(data_path, encoding="utf-8-sig") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [ ]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [ ]:
# 1. Выручка после очистки
months = np.array(["2026-09", "2026-10"])
monthly_revenue = np.array([amount[clean & (month == m)].sum() for m in months])
revenue_change = (monthly_revenue[1] / monthly_revenue[0] - 1) * 100
print(f"Чистых заказов: {clean.sum():,}")
print(f"Сентябрь: {monthly_revenue[0]:,.0f} ₸; октябрь: {monthly_revenue[1]:,.0f} ₸")
print(f"Изменение: {revenue_change:.2f}% ({monthly_revenue[1] - monthly_revenue[0]:,.0f} ₸)")
print(f"Менеджеру: после очистки выручка снизилась на {abs(revenue_change):.2f}%.")
print("Результат согласуется с ориентиром ДЗ 2 из урока: около −2,3%; сам файл ДЗ 2 здесь не приложен.")

Чистых заказов: 4,739
Сентябрь: 35,243,410 ₸; октябрь: 34,419,520 ₸
Изменение: -2.34% (-823,890 ₸)
Менеджеру: после очистки выручка снизилась на 2.34%.
Результат согласуется с ориентиром ДЗ 2 из урока: около −2,3%; сам файл ДЗ 2 здесь не приложен.


## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [ ]:
# 2. Выручка: категория × месяц
categories = np.unique(cat)
category_revenue = np.array([[amount[clean & (cat == c) & (month == m)].sum()
                              for m in months] for c in categories])
category_totals = category_revenue.sum(axis=1)
category_change = (category_revenue[:, 1] / category_revenue[:, 0] - 1) * 100
print(f"{'Категория':<16} {'Сентябрь, ₸':>15} {'Октябрь, ₸':>15} {'Изменение, %':>14}")
for c, r, change in zip(categories, category_revenue, category_change):
    print(f"{c:<16} {r[0]:>15,.0f} {r[1]:>15,.0f} {change:>14.2f}")
leader = category_totals.argmax()
worst = category_change.argmin()
print(f"Форма таблицы: {category_revenue.shape}")
print(f"Менеджеру: лидер за два месяца — {categories[leader]} ({category_totals[leader]:,.0f} ₸); "
      f"сильнее всего снизилась категория {categories[worst]} ({category_change[worst]:.2f}%).")

Категория            Сентябрь, ₸      Октябрь, ₸   Изменение, %
Дом                    4,997,730       5,125,790           2.56
Книги                  1,065,300       1,140,910           7.10
Красота                2,610,950       2,678,990           2.61
Одежда                 9,167,100       8,505,730          -7.21
Электроника           17,402,330      16,968,100          -2.50
Форма таблицы: (5, 2)
Менеджеру: лидер за два месяца — Электроника (34,370,430 ₸); сильнее всего снизилась категория Одежда (-7.21%).


## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [ ]:
# 3. Количество заказов и средний чек
category_counts = np.array([[(clean & (cat == c) & (month == m)).sum()
                             for m in months] for c in categories])
average_check = category_revenue / category_counts
print(f"{'Категория':<16} {'Заказов, сен.':>13} {'Заказов, окт.':>13} {'Чек сен., ₸':>14} {'Чек окт., ₸':>14}")
for c, count, avg in zip(categories, category_counts, average_check):
    print(f"{c:<16} {count[0]:>13} {count[1]:>13} {avg[0]:>14,.2f} {avg[1]:>14,.2f}")
best_check = average_check[:, 1].argmax()
print(f"Менеджеру: самый высокий средний чек октября — {categories[best_check]}, "
      f"{average_check[best_check, 1]:,.2f} ₸.")

Категория        Заказов, сен. Заказов, окт.    Чек сен., ₸    Чек окт., ₸
Дом                        476           476      10,499.43      10,768.47
Книги                      265           299       4,020.00       3,815.75
Красота                    422           435       6,187.09       6,158.60
Одежда                     678           652      13,520.80      13,045.60
Электроника                531           505      32,772.75      33,600.20
Менеджеру: самый высокий средний чек октября — Электроника, 33,600.20 ₸.


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [ ]:
# 4. Самостоятельно собираем таблицу городов из урока
cities = np.unique(city)
table = np.array([[amount[clean & (city == c) & (month == m)].sum()
                   for m in months] for c in cities])
city_shares = table / table.sum(axis=0)
print(f"{'Город':<12} {'Сентябрь, %':>13} {'Октябрь, %':>13}")
for c, share in zip(cities, city_shares):
    print(f"{c:<12} {share[0]*100:>13.2f} {share[1]*100:>13.2f}")
print("Суммы долей по столбцам:", city_shares.sum(axis=0))
assert np.allclose(city_shares.sum(axis=0), 1)
shymkent = np.flatnonzero(cities == "Шымкент")[0]
share_delta_pp = (city_shares[shymkent, 1] - city_shares[shymkent, 0]) * 100
print(f"Менеджеру: доля Шымкента снизилась с {city_shares[shymkent, 0]*100:.2f}% "
      f"до {city_shares[shymkent, 1]*100:.2f}% ({share_delta_pp:.2f} п.п.).")

Город          Сентябрь, %    Октябрь, %
Актобе                7.31          7.38
Алматы               41.15         42.54
Астана               25.63         25.71
Атырау                2.62          2.82
Караганда            10.35          9.80
Шымкент              12.93         11.75
Суммы долей по столбцам: [1. 1.]
Менеджеру: доля Шымкента снизилась с 12.93% до 11.75% (-1.19 п.п.).


## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [ ]:
# 5. Вклад в общее изменение выручки
city_delta = table[:, 1] - table[:, 0]
contribution = city_delta / city_delta.sum()
print(f"{'Город':<12} {'Изменение, ₸':>16} {'Вклад в падение, %':>20}")
for c, delta, share in zip(cities, city_delta, contribution):
    print(f"{c:<12} {delta:>16,.0f} {share*100:>20.2f}")
print(f"Сумма вкладов: {contribution.sum()*100:.2f}%")
print("Отрицательный вклад означает рост города, который частично компенсировал общее падение.")
print(f"Менеджеру: Шымкент даёт {contribution[shymkent]*100:.2f}% общего падения "
      f"({-city_delta[shymkent]:,.0f} ₸); в первую очередь проверяем причины снижения продаж в этом городе.")

Город            Изменение, ₸   Вклад в падение, %
Актобе                -36,540                 4.44
Алматы                141,950               -17.23
Астана               -183,690                22.30
Атырау                 45,870                -5.57
Караганда            -276,840                33.60
Шымкент              -514,640                62.46
Сумма вкладов: 100.00%
Отрицательный вклад означает рост города, который частично компенсировал общее падение.
Менеджеру: Шымкент даёт 62.46% общего падения (514,640 ₸); в первую очередь проверяем причины снижения продаж в этом городе.


## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [ ]:
# 6. Исправление broadcasting
print("table имеет форму", table.shape, "; сумма строк без keepdims —", table.sum(axis=1).shape)
print("Размеры сравниваются справа: 2 и 6 несовместимы; нужен столбец итогов формы (6, 1).")
share = table / table.sum(axis=1, keepdims=True)
print(f"{'Город':<12} {'Сентябрь, %':>13} {'Октябрь, %':>13}")
for c, row in zip(cities, share):
    print(f"{c:<12} {row[0]*100:>13.2f} {row[1]*100:>13.2f}")
print("Суммы долей по строкам:", share.sum(axis=1))
assert np.allclose(share.sum(axis=1), 1)
print("Менеджеру: для каждого из 6 городов доли двух месяцев в его общей выручке дают 100%.")

table имеет форму (6, 2) ; сумма строк без keepdims — (6,)
Размеры сравниваются справа: 2 и 6 несовместимы; нужен столбец итогов формы (6, 1).
Город          Сентябрь, %    Октябрь, %
Актобе               50.36         49.64
Алматы               49.76         50.24
Астана               50.51         49.49
Атырау               48.79         51.21
Караганда            51.97         48.03
Шымкент              52.99         47.01
Суммы долей по строкам: [1. 1. 1. 1. 1. 1.]
Менеджеру: для каждого из 6 городов доли двух месяцев в его общей выручке дают 100%.


## [Челлендж] 7. Стандартизация суммы заказа
Для чистых оплаченных заказов посчитайте z-оценку: `z = (x - x.mean()) / x.std()`.
Сколько заказов с |z| > 3 и какая это доля? Проверьте, что у `z` среднее ≈ 0 и std = 1.

In [ ]:
# 7. Z-оценка; np.std по умолчанию использует ddof=0
x = amount[clean]
z = (x - x.mean()) / x.std()
outliers = np.abs(z) > 3
print(f"Среднее суммы: {x.mean():,.2f} ₸; стандартное отклонение: {x.std():,.2f} ₸")
print(f"Заказов с |z| > 3: {outliers.sum()}; доля: {outliers.mean()*100:.2f}%")
print(f"Среднее z: {z.mean():.3e}; std(z): {z.std():.12f}")
assert np.isclose(z.mean(), 0, atol=1e-12) and np.isclose(z.std(), 1)
print(f"Менеджеру: {outliers.sum()} заказов ({outliers.mean()*100:.2f}%) "
      "существенно отличаются по сумме; это повод изучить их, а не автоматически удалять.")

Среднее суммы: 14,699.92 ₸; стандартное отклонение: 13,291.56 ₸
Заказов с |z| > 3: 87; доля: 1.84%
Среднее z: 7.047e-17; std(z): 1.000000000000
Менеджеру: 87 заказов (1.84%) существенно отличаются по сумме; это повод изучить их, а не автоматически удалять.


## [Челлендж] 8. Масштабирование признаков (заготовка для ML)
Соберите матрицу `X` из двух столбцов: `amount` и `days` для чистых заказов без пропусков в `days` (`np.column_stack`).
Приведите каждый столбец к диапазону [0, 1] одной строкой: `(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))`.
Проверьте min и max каждого столбца. Объясните своими словами, зачем модели нужны признаки в одном масштабе.

In [ ]:
# 8. Масштабирование двух признаков
valid = clean & ~np.isnan(days)
X = np.column_stack((amount[valid], days[valid]))
X_scaled = (X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))
print("Форма X:", X.shape)
print("Исключено чистых заказов без срока доставки:", clean.sum() - valid.sum())
print("Исходные минимумы [amount, days]:", X.min(axis=0))
print("Исходные максимумы [amount, days]:", X.max(axis=0))
print("Минимумы после масштабирования:", X_scaled.min(axis=0))
print("Максимумы после масштабирования:", X_scaled.max(axis=0))
assert np.allclose(X_scaled.min(axis=0), 0) and np.allclose(X_scaled.max(axis=0), 1)
print(f"Менеджеру: подготовлены {X.shape[0]} заказов с 2 признаками в диапазоне [0, 1].")
print("Единый масштаб не даёт суммам в тенге подавлять сроки в днях в моделях, чувствительных "
      "к расстояниям и масштабу, и помогает обучению градиентными методами. "
      "Для деревьев это обычно не требуется. В ML минимумы и максимумы определяют "
      "на обучающей выборке и применяют к остальным без повторной подгонки.")

Форма X: (4595, 2)
Исключено чистых заказов без срока доставки: 144
Исходные минимумы [amount, days]: [720.   1.]
Исходные максимумы [amount, days]: [1.3904e+05 8.0000e+00]
Минимумы после масштабирования: [0. 0.]
Максимумы после масштабирования: [1. 1.]
Менеджеру: подготовлены 4595 заказов с 2 признаками в диапазоне [0, 1].
Единый масштаб не даёт суммам в тенге подавлять сроки в днях в моделях, чувствительных к расстояниям и масштабу, и помогает обучению градиентными методами. Для деревьев это обычно не требуется. В ML минимумы и максимумы определяют на обучающей выборке и применяют к остальным без повторной подгонки.
